In [1]:
# Modules de base
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Module relatif à Gurobi
from gurobipy import *

## Load files

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re
from typing import Any, Dict, List, Tuple, Optional, Union


# ----------------------------
# 1) File locator (decoupled)
# ----------------------------

def _format_density(density: Union[int, float, str]) -> str:
    """
    Convert density to the exact string used in folder/file names.
    Examples:
      1    -> "1"
      0.9  -> "0.9"
      0.95 -> "0.95"
      "4"  -> "4"
    """
    if isinstance(density, str):
        s = density.strip()
        # Normalize "1.0" -> "1" etc.
        try:
            f = float(s)
            if f.is_integer():
                return str(int(f))
            return s.rstrip("0").rstrip(".") if "." in s else s
        except ValueError:
            return s

    if isinstance(density, (int, float)):
        f = float(density)
        if f.is_integer():
            return str(int(f))
        # keep minimal representation (0.90 -> 0.9)
        s = repr(f)
        return s.rstrip("0").rstrip(".")
    return str(density)


def find_dat_file(
    base_dir: Union[str, Path],
    density: Union[int, float, str],
    p: int,
    h: int,
    test: Optional[int] = None,
    *,
    top_folders: Tuple[str, str] = ("Data", "Data with maintenance constraints"),
    folder_prefix: str = "d=",
    file_prefix: str = "DataCplex_",
) -> Path:
    """
    Search for the .dat matching density/p/h/(optional)test under:
      base_dir/Data/d=<density>/
      base_dir/Data with maintenance constraints/d=<density>/
    Returns the Path if uniquely found, otherwise raises an error.
    """
    base_dir = Path(base_dir)
    d_str = _format_density(density)

    candidates: List[Path] = []

    # Search both top folders (works even if one doesn't exist)
    for top in top_folders:
        d_folder = base_dir / top / f"{folder_prefix}{d_str}"
        if not d_folder.exists():
            continue

        if test is None:
            pattern = f"{file_prefix}density={d_str}_p={p}_h={h}_test_*.dat"
        else:
            pattern = f"{file_prefix}density={d_str}_p={p}_h={h}_test_{test}.dat"

        candidates.extend(sorted(d_folder.glob(pattern)))

    if not candidates:
        raise FileNotFoundError(
            f"No .dat file found for density={d_str}, p={p}, h={h}, test={test} under {base_dir}"
        )

    # If test=None we might match multiple; require uniqueness to be safe
    if len(candidates) > 1:
        msg = "\n".join(str(c) for c in candidates[:30])
        more = "" if len(candidates) <= 30 else f"\n... and {len(candidates)-30} more"
        raise FileExistsError(
            f"Multiple matching .dat files found for density={d_str}, p={p}, h={h}, test={test}:\n{msg}{more}\n"
            f"Pass test=<n> to select a unique file."
        )

    return candidates[0]


# ----------------------------
# 2) Parser (decoupled)
# ----------------------------

_DECL_RE = re.compile(r"(?ms)^\s*(\w+)\s*=\s*(.*?)\s*;\s*$")

def _parse_atom(token: str) -> Union[int, float, str]:
    token = token.strip()
    if token == "":
        return token

    # Try int then float
    try:
        if re.fullmatch(r"[+-]?\d+", token):
            return int(token)
        if re.fullmatch(r"[+-]?\d*\.\d+(?:[eE][+-]?\d+)?", token) or re.fullmatch(r"[+-]?\d+(?:[eE][+-]?\d+)", token):
            return float(token)
    except ValueError:
        pass

    # Otherwise string (e.g., airport codes A,B,C)
    return token


def _parse_set(value: str) -> List[Union[int, float, str]]:
    # value like "{A,B,C,}" or "{0,1,2,}"
    inner = value.strip()[1:-1].strip()
    if inner == "":
        return []
    parts = [p.strip() for p in inner.split(",")]
    parts = [p for p in parts if p != ""]
    return [_parse_atom(p) for p in parts]


def _parse_tuples_from_angle_brackets(value: str) -> List[Tuple[Union[int, float, str], ...]]:
    # Extract all <...> blocks, split by commas inside
    tuples: List[Tuple[Union[int, float, str], ...]] = []
    for body in re.findall(r"<([^<>]*)>", value, flags=re.S):
        fields = [f.strip() for f in body.split(",")]
        fields = [f for f in fields if f != ""]
        tuples.append(tuple(_parse_atom(f) for f in fields))
    return tuples


def _parse_matrix(value: str) -> List[List[Union[int, float]]]:
    """
    Parses something like:
      Cost =[
        [6804.0,6870.0,...,]
        [4536.0,4580.0,...,]
      ];
    Note: rows are NOT comma-separated in your file, so we scan bracket depth.
    """
    s = value.strip()
    if not (s.startswith("[") and s.endswith("]")):
        raise ValueError("Matrix must start with '[' and end with ']'")

    content = s[1:-1].strip()

    rows: List[str] = []
    i = 0
    n = len(content)
    while i < n:
        # Find next row '['
        while i < n and content[i].isspace():
            i += 1
        if i >= n:
            break
        if content[i] != "[":
            # skip unexpected chars
            i += 1
            continue

        # parse one row [...]
        depth = 0
        start = i
        while i < n:
            ch = content[i]
            if ch == "[":
                depth += 1
            elif ch == "]":
                depth -= 1
                if depth == 0:
                    end = i
                    rows.append(content[start + 1 : end])  # inside [...]
                    i += 1
                    break
            i += 1

    matrix: List[List[Union[int, float]]] = []
    for row in rows:
        nums = [x.strip() for x in row.replace("\n", " ").split(",")]
        nums = [x for x in nums if x != ""]
        parsed_row: List[Union[int, float]] = []
        for x in nums:
            atom = _parse_atom(x)
            if isinstance(atom, str):
                raise ValueError(f"Non-numeric value in matrix: {atom!r}")
            parsed_row.append(atom)
        matrix.append(parsed_row)

    return matrix


def parse_dat_file(dat_path: Union[str, Path]) -> Dict[str, Any]:
    """
    Parses an OPL/CPLEX-like .dat into a dict:
      - Sets {...} -> Python list
      - Scalars -> int/float/str
      - Tuple lists <...> -> list[tuple]
      - Matrices [...] with row brackets -> list[list[float/int]]
    """
    dat_path = Path(dat_path)
    text = dat_path.read_text(encoding="utf-8", errors="replace")

    out: Dict[str, Any] = {}

    # Split by semicolons safely by matching "name = ... ;" blocks (multiline)
    for m in _DECL_RE.finditer(text):
        name = m.group(1)
        raw = m.group(2).strip()

        if raw.startswith("{") and raw.endswith("}"):
            # could be a set, or a tuple-set like Flight = { <...> <...> }
            if "<" in raw and ">" in raw:
                out[name] = _parse_tuples_from_angle_brackets(raw)
            else:
                out[name] = _parse_set(raw)

        elif raw.startswith("[") and raw.endswith("]"):
            # could be matrix, or list of tuples inside brackets
            if "<" in raw and ">" in raw:
                out[name] = _parse_tuples_from_angle_brackets(raw)
            else:
                out[name] = _parse_matrix(raw)

        else:
            # scalar
            out[name] = _parse_atom(raw)

    return out



In [3]:
base = Path.cwd()  # or Path(__file__).resolve().parent

# Find file by params:
dat_file = find_dat_file(base, density=0.9, p=10, h=7, test=0)

data = parse_dat_file(dat_file)

print("Keys:", data.keys())
print("Nbflight:", data["Nbflight"])
print("First Flight tuple:", data["Flight"][0])
print("Cost shape:", len(data["Cost"]), "x", len(data["Cost"][0]) if data["Cost"] else 0)
print("Initial aircraft position tuple:", data["Aircraft"][0], " (Aircraft, 'Airport')")

Airports   = data["Airports"] # list
Nbflight   = data["Nbflight"] # int
Aircrafts  = data["Aircrafts"] # list
Flights    = data["Flight"] # list of tuples
Cost       = data["Cost"] # matrix
InitialPositions   = data["Aircraft"] # list of tuples

Keys: dict_keys(['Airports', 'Nbflight', 'Aircrafts', 'Flight', 'Cost', 'Aircraft'])
Nbflight: 167
First Flight tuple: (1, 'MAD', 'HAM', 6360.0, 6615.0)
Cost shape: 167 x 10
Initial aircraft position tuple: (0, 'HAM')  (Aircraft, 'Airport')


## Add virtual flights

In [4]:
# Create list of flights with 'virtual' first flights to enable initial position
# + create list of flights with 'virtual' end flight to ensure end of the condition
virtual_flights_id = -1 # start counting the virtual flight IDs in negative to not interfere with the real flights' IDs
Original_Flights = Flights.copy()#[] # keep the original flights in a list
#Original_Flights.extend(Flights)

# First flights
First_Flights = []
for initial_position_tuple in InitialPositions:
    First_Flights.append((virtual_flights_id, '', initial_position_tuple[1], 0, 0))
    virtual_flights_id = virtual_flights_id + 1
Flights.extend(First_Flights)
    
# End flights for each airport
for airport in Airports:
    Flights.append((virtual_flights_id, airport, '', 999999999, 0))
    virtual_flights_id = virtual_flights_id + 1

## Create matrix of possible flight combinations

In [5]:
# Create matrix A_ij with possible flight combinations
# A[i][j] will check if the flight j can happen after flight i
A = []
for flight in Flights:
    row = []
    for flight_compare in Flights:
        # check if 'flight_compare' can happen after 'flight'
        # this is possible if the departure time of the second flight is greater than the arrival time of the first flight
        # and if the departure airport of the second flight is equal to the arrival airport of the first flight
        if flight_compare[3] >= flight[4] and flight_compare[1] == flight[2]:
            row.append(1)
        else:
            row.append(0)
    # Add the row to the matrix
    A.append(row)


## Build x_ijt model

In [6]:
# CREATE MODEL
m = Model("x_ijt")

# ADD VARIABLES
Flight_IDs = [f[0] for f in Flights]
X = {}#m.addVars(Flight_IDs, Flight_IDs, Aircrafts, vtype=GRB.BINARY, name = "x")
for flight_i in Flight_IDs:
    for flight_j in Flight_IDs:
        if not flight_i == flight_j:
            for aircraft_t in Aircrafts:
                X[(flight_i, flight_j, aircraft_t)] = m.addVar(vtype=GRB.BINARY, name = f'x_{flight_i}_{flight_j}_{aircraft_t}')
#X = {(i, j, t) :  m.addVar(vtype = GRB.BINARY, name = f'x_{i}_{j}_{t}') for i in Flights for j in Flights for t in Aircrafts}

# ADD CONSTRAINTS

# Only allow combinations of flights represented in matrix A
m.addConstrs(
    (X[(i, j, t)] <= A[i][j]
     for i in Flight_IDs
     for j in Flight_IDs if not i == j
     for t in Aircrafts),
    name="CombinationsMatrixCondition"
)

# Only one plane per flight
m.addConstrs(
    (quicksum(
        quicksum(
            X[(i, j, t)]
            for i in Flight_IDs if not i == j)
        for t in Aircrafts)
     == 1
     for j in Flight_IDs),
    name = "OnePlanePerFlightCondition")

# Continuity of the flights
m.addConstrs(
    (quicksum(
        X[(i, j, t)]
        for i in Flight_IDs if not i == j)
     ==
     quicksum(
        X[(j, k, t)]
        for k in Flight_IDs if not j == k)
     for j in Flight_IDs
     for t in Aircrafts),
    name = "FlightsContinuityCondition"
)

# Initial position condition
# Each aircraft must have one flight at the beginning
for first in First_Flights:
    m.addConstr(
        quicksum(
            X[(i, j, t)]
            for j in Flight_IDs
                for i in Flight_IDs
                if i is first and not j == i
                    for t in Aircrafts
                    if t == first[0])
        == 1,
        name = "InitialPositionCondition_{first[0]}")

# Add objective function
Original_Flights_IDs = [f[0] for f in Original_Flights]
m.setObjective(
    quicksum(
        quicksum(
            quicksum(
                X[(i, j, t)] * Cost[j-1][t] # j-1 because flight IDs start at 1 and array index starts at 0
                for j in Original_Flights_IDs if not i == j)
            for i in Original_Flights_IDs)
        for t in Aircrafts),
    GRB.MINIMIZE)

Set parameter Username
Set parameter LicenseID to value 2773683
Academic license - for non-commercial use only - expires 2027-02-02


## Update and store model

In [7]:
# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet
# - 

# -- Mise à jour du modèle  --
m.update()

# Save model into file
m.write("x_ijt_model.lp")

# Optimize
m.optimize()

In [8]:
if m.status == GRB.OPTIMAL:
    print("\n" + "="*40)
    print(f"--- OPTIMAL SCHEDULE FOUND ---")
    print(f"Total Operational Cost: {m.objVal:,.2f}")
    print("="*40)
elif m.status == GRB.INFEASIBLE:
    print("\nNo solution found (INFEASIBLE).")
    print("Computing IIS to find the exact conflicting constraints...")
    
    # This will calculate the conflict and write it to a file
    m.computeIIS()
    m.write("conflict_model.ilp")
    
    print("\nCheck the 'conflict_model.ilp' file generated in your folder.")
    print("It will show you exactly which flights/aircraft rules are impossible to satisfy together.")
    
else:
    print(f"\nSolver finished with status code: {m.status}")


No solution found (INFEASIBLE).
Computing IIS to find the exact conflicting constraints...

Check the 'conflict_model.ilp' file generated in your folder.
It will show you exactly which flights/aircraft rules are impossible to satisfy together.


## Display

In [9]:
# --------- Build assigned flights from x[i,j,t] ----------

assigned = {t: [] for t in Aircrafts}
airports = set()

# Helper: map flight_id -> full flight tuple
# Assuming Flights is a list of tuples: (id, origin, dest, dep, arr)
FlightInfo = {f[0]: f for f in Flights}

# 1. Find all predecessors for each aircraft
predecessor = {t: {} for t in Aircrafts}
successor   = {t: {} for t in Aircrafts}

for i in Flight_IDs:
    for j in Flight_IDs:
        for t in Aircrafts:
            if (i, j, t) in X and X[i, j, t].x > 0.5:
                successor[t][i] = j
                predecessor[t][j] = i

# 2. For each aircraft, find the starting flight (no predecessor)
for t in Aircrafts:
    # all flights that appear as i or j for this aircraft
    flights_t = set(successor[t].keys()) | set(predecessor[t].keys())
    if not flights_t:
        continue

    start_candidates = [f for f in flights_t if f not in predecessor[t]]
    if len(start_candidates) != 1:
        # If model is correct, this should be exactly one
        continue

    start = start_candidates[0]

    # 3. Follow the chain i -> j -> k -> ...
    seq = []
    current = start
    while current in successor[t]:
        seq.append(current)
        current = successor[t][current]
    seq.append(current)  # last flight

    # 4. Convert flight IDs to (dep, arr, id, origin, dest)
    for fid in seq:
        fid, origin, dest, dep, arr = FlightInfo[fid]
        dep = float(dep)
        arr = float(arr)
        assigned[t].append((dep, arr, fid, origin, dest))
        airports.add(origin)
        airports.add(dest)

# 5. Sort flights per aircraft by departure time
for t in Aircrafts:
    assigned[t].sort(key=lambda x: x[0])

# --------- Assign one color per airport ----------
airports = sorted(list(airports))
cmap = plt.get_cmap("tab20", len(airports))
airport_color = {ap: cmap(idx) for idx, ap in enumerate(airports)}

# --------- Plot ----------
row_spacing = 1.4   # increase spacing between aircraft rows
bar_height  = 0.7   # slightly taller bars

# Count used aircraft to set figure height properly
used_aircraft = sum(1 for k in Aircrafts if assigned[k])
fig, ax = plt.subplots(figsize=(13, max(2, 0.45 * used_aircraft)))

y = 0
yticks = []
ylabels = []

for k_id in Aircrafts:
    # Skip plotting if the aircraft has no assigned flights
    if not assigned[k_id]:
        continue

    yticks.append(y)
    # Use the original aircraft ID
    ylabels.append(f"AC {k_id}")

    for (dep, arr, fid, origin, dest) in assigned[k_id]:
        duration = arr - dep
        mid = dep + duration / 2.0

        # left half = origin color
        ax.barh(
            y, mid - dep, left=dep, height=bar_height,
            color=airport_color[origin],
            edgecolor="black", linewidth=0.9
        )

        # right half = destination color
        ax.barh(
            y, arr - mid, left=mid, height=bar_height,
            color=airport_color[dest],
            edgecolor="black", linewidth=0.9
        )

        # flight ID ABOVE the block (centered)
        ax.text(
            dep + duration/2,
            y + bar_height/2 + 0.08,
            str(fid),
            ha="center",
            va="bottom",
            fontsize=7,
            fontweight="bold"
        )

    y += row_spacing

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels)
ax.set_xlabel("Time (minutes from midnight)")
ax.set_title("Assigned Flights per Aircraft (Origin/Destination Colors)")

# Add a legend for the airports
handles = [plt.Rectangle((0,0),1,1, color=airport_color[ap]) for ap in airports]
ax.legend(handles, airports, title="Airports", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

AttributeError: Unable to retrieve attribute 'x'